# Factual Recall γ — Phase 0: Data Preparation + GPU Smoke Tests

**Part A (CPU):** COUNTERFACT → `trial_definitions.csv`, `competitor_map.csv`, `relation_manifest.csv`
**Part B (GPU):** smoke-test Llama-standard per-head patching pattern on Llama-8B-Inst.

GPT-J / Gemma-3 patterns are printed at the end as reference (run in their own notebooks).

Output: `{DRIVE_BASE}/factual_recall/00_data/`.


In [ ]:
# ── Cell 1: Drive mount → pip install → HF login ──
from google.colab import drive, runtime
drive.mount('/content/drive')

!pip install -q nnsight transformers accelerate scipy pandas scikit-learn datasets

import os
os.environ['HF_TOKEN'] = '<YOUR_HF_TOKEN>'
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
from huggingface_hub import login as _hf_login
_hf_login(token=os.environ['HF_TOKEN'])
print('HF_TOKEN set + huggingface_hub.login() called.')

import json, gc, time, math, random
from collections import defaultdict, Counter
from datetime import datetime
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

def log(msg):
    print(f'[{datetime.now().strftime("%H:%M:%S")}] {msg}', flush=True)

log('Imports + HF login complete.')


In [ ]:
# ── Cell 2: Config ──
# Tokenizer IDs (gated first for fast auth-failure surfacing).
# For Gemma-3: use google/gemma-3-27b-it for tokenizer; GPU notebook loads gghfez/gemma-3-27b-novision for weights.
MODEL_NAMES = [
    'meta-llama/Llama-3.1-8B-Instruct',
    'cmarkea/Meta-Llama-3.1-70B-Instruct-4bit',  # 70B-NF4 tokenizer (verify identity vs 8B per Llama-3.1 family spec)
    'google/gemma-2-9b-it',
    'google/gemma-3-27b-it',
    'Qwen/Qwen2.5-7B-Instruct',
    'Qwen/Qwen2.5-14B-Instruct',
    'allenai/OLMo-2-1124-13B-Instruct',
    'EleutherAI/gpt-j-6b',
]

DRIVE_BASE = '/content/drive/MyDrive/WCC'
OUT_DIR    = f'{DRIVE_BASE}/factual_recall/00_data'
os.makedirs(OUT_DIR, exist_ok=True)

MIN_TRIALS_PER_RELATION = 5
SEED = 42
random.seed(SEED); np.random.seed(SEED)

log(f'Output directory: {OUT_DIR}')


---
## Part A — Data Preparation (CPU)


In [ ]:
# ── Cell 3: Download COUNTERFACT from rome.baulab.info ──
COUNTERFACT_URL = 'https://rome.baulab.info/data/dsets/counterfact.json'
LOCAL_CF_PATH = '/content/counterfact.json'

if not os.path.exists(LOCAL_CF_PATH):
    log(f'Downloading COUNTERFACT from {COUNTERFACT_URL}')
    urlretrieve(COUNTERFACT_URL, LOCAL_CF_PATH)
    log(f'  {os.path.getsize(LOCAL_CF_PATH)/1024/1024:.1f} MB downloaded')
else:
    log(f'COUNTERFACT cached: {LOCAL_CF_PATH}')

with open(LOCAL_CF_PATH) as f:
    cf_raw = json.load(f)

log(f'COUNTERFACT entries: {len(cf_raw)}')


In [ ]:
# ── Cell 4: Helper functions ──

def is_single_token_with_space(tokenizer, text: str) -> bool:
    if not isinstance(text, str) or text == '':
        return False
    prefix = 'The answer is'
    try:
        full = tokenizer.encode(prefix + ' ' + text, add_special_tokens=False)
        pref = tokenizer.encode(prefix, add_special_tokens=False)
    except Exception:
        return False
    return len(full) - len(pref) == 1

def get_token_id_with_space(tokenizer, text: str) -> int:
    prefix = 'The answer is'
    full = tokenizer.encode(prefix + ' ' + text, add_special_tokens=False)
    pref = tokenizer.encode(prefix, add_special_tokens=False)
    return full[len(pref)]

def check_all_tokenizers(tokenizers: dict, text: str) -> bool:
    return all(is_single_token_with_space(tok, text) for tok in tokenizers.values())

log('Helper functions defined.')


In [ ]:
# ── Cell 5: Load 7 tokenizers ──
from transformers import AutoTokenizer

tokenizers = {}
for name in MODEL_NAMES:
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True, token=os.environ['HF_TOKEN'])
    tokenizers[name] = tok
    log(f'  loaded {name}  (vocab={tok.vocab_size}, {time.time()-t0:.1f}s)')

log(f'All {len(tokenizers)} tokenizers loaded.')
for name, tok in tokenizers.items():
    log(f'  {name}: "Paris" single-token = {is_single_token_with_space(tok, "Paris")}')

# === Identity check: 8B vs 70B Llama-3.1 tokenizer ===
_id_8b  = 'meta-llama/Llama-3.1-8B-Instruct'
_id_70b = 'cmarkea/Meta-Llama-3.1-70B-Instruct-4bit'
if _id_8b in tokenizers and _id_70b in tokenizers:
    _t8  = tokenizers[_id_8b]
    _t70 = tokenizers[_id_70b]
    _vocab_match = (_t8.vocab_size == _t70.vocab_size)
    _spec_match = (sorted(_t8.all_special_tokens) == sorted(_t70.all_special_tokens))
    # Encode the same probe set both ways and compare token IDs
    _probes = ['Paris', 'France', 'Tokyo', 'Berlin', 'London', 'piano', 'soccer']
    _enc_match = all(
        _t8.encode('The answer is ' + p, add_special_tokens=False)
            == _t70.encode('The answer is ' + p, add_special_tokens=False)
        for p in _probes
    )
    log(f'  Llama-3.1 8B vs 70B tokenizer: vocab_match={_vocab_match} '
        f'special_match={_spec_match} probe_encode_match={_enc_match}')
    assert _vocab_match and _spec_match and _enc_match, (
        f'Llama-3.1 8B/70B tokenizer divergence detected — '
        f'vocab={_vocab_match} special={_spec_match} probe={_enc_match}. '
        f'Phase 0 filter may not be valid for 70B; STOP and investigate.'
    )
    log('  → 8B/70B tokenizer identity confirmed; Phase 0 filter is valid for 70B.')


In [ ]:
# ── Cell 6: Filter COUNTERFACT by single-token target across all 7 tokenizers ──
t0 = time.time()
relation_trials = defaultdict(list)
for idx, entry in enumerate(cf_raw):
    rr = entry.get('requested_rewrite', {})
    relation, subject = rr.get('relation_id'), rr.get('subject')
    prompt_tpl = rr.get('prompt')
    tt = rr.get('target_true')
    target = tt.get('str') if isinstance(tt, dict) else None
    if relation is None or subject is None or target is None or prompt_tpl is None:
        continue
    if '{}' not in prompt_tpl:
        continue
    try:
        prompt = prompt_tpl.format(subject)
    except Exception:
        continue
    if not check_all_tokenizers(tokenizers, target):
        continue
    relation_trials[relation].append({
        'case_id' : entry.get('case_id', idx),
        'relation': relation, 'subject': subject, 'target': target, 'prompt': prompt,
    })

log(f'After single-token target filter: {sum(len(v) for v in relation_trials.values())} trials '
    f'across {len(relation_trials)} relations ({time.time()-t0:.1f}s)')

valid_relations = {rel: tl for rel, tl in relation_trials.items() if len(tl) >= MIN_TRIALS_PER_RELATION}
log(f'Relations with ≥{MIN_TRIALS_PER_RELATION} trials: {len(valid_relations)}')

# Competitor pool per relation, ordered by frequency (most common target first)
competitor_map = {}
for relation, tl in valid_relations.items():
    tgt_counts = Counter(t['target'] for t in tl)
    ordered = [t for t, _ in tgt_counts.most_common() if check_all_tokenizers(tokenizers, t)]
    if len(ordered) >= 2:
        competitor_map[relation] = {'targets': ordered, 'n_trials': len(tl)}
log(f'Relations with competitor pool (≥ 2 single-token targets): {len(competitor_map)}')


In [ ]:
# ── Cell 7: Assign corrupt_prompts (circular within relation) + finalize ──
final_trials = []
for relation, tl in valid_relations.items():
    if relation not in competitor_map:
        continue
    rel_trials = sorted(tl, key=lambda t: t['case_id'])
    pool = competitor_map[relation]['targets']
    n = len(rel_trials)
    for i, trial in enumerate(rel_trials):
        corrupt_trial = rel_trials[(i + 1) % n]
        # Competitor = most-frequent target in same relation that isn't trial's own target (§3.5).
        others = [t for t in pool if t != trial['target']]
        if not others:
            continue
        competitor = others[0]
        final_trials.append({
            'trial_id'       : f'{relation}_{trial["case_id"]}',
            'relation'       : relation,
            'subject'        : trial['subject'],
            'target'         : trial['target'],
            'competitor'     : competitor,
            'prompt'         : trial['prompt'],
            'corrupt_subject': corrupt_trial['subject'],
            'corrupt_target' : corrupt_trial['target'],
            'corrupt_prompt' : corrupt_trial['prompt'],
        })

log(f'Final trials: {len(final_trials)} across {len({t["relation"] for t in final_trials})} relations')


In [ ]:
# ── Cell 8: Save 3 CSVs + config.json + print stats ──
trial_df = pd.DataFrame(final_trials)
trial_df.to_csv(f'{OUT_DIR}/trial_definitions.csv', index=False)
log(f'Wrote trial_definitions.csv ({len(trial_df)} rows)')

rel_counts = trial_df.groupby('relation').agg(
    n_trials=('trial_id', 'count'),
    n_unique_targets=('target', 'nunique'),
    n_unique_subjects=('subject', 'nunique'),
).reset_index()
rel_counts.to_csv(f'{OUT_DIR}/relation_manifest.csv', index=False)
log(f'Wrote relation_manifest.csv ({len(rel_counts)} relations)')

comp_rows = [{'relation': r, 'target': t} for r, info in competitor_map.items() for t in info['targets']]
pd.DataFrame(comp_rows).to_csv(f'{OUT_DIR}/competitor_map.csv', index=False)
log(f'Wrote competitor_map.csv ({len(comp_rows)} rows)')

with open(f'{OUT_DIR}/config.json', 'w') as f:
    json.dump({
        'seed'                   : SEED,
        'model_names'            : MODEL_NAMES,
        'counterfact_url'        : COUNTERFACT_URL,
        'min_trials_per_relation': MIN_TRIALS_PER_RELATION,
        'n_final_trials'         : len(trial_df),
        'n_final_relations'      : int(trial_df['relation'].nunique()) if len(trial_df) else 0,
        'completed_at'           : datetime.now().isoformat(),
    }, f, indent=2)

log('')
log(f'Total trials  : {len(trial_df)}')
log(f'Relations     : {trial_df["relation"].nunique() if len(trial_df) else 0}')
print()
print('Top relations:')
print(rel_counts.sort_values('n_trials', ascending=False).head(10).to_string(index=False))


---
## Part B — GPU Smoke Test

Exercises the Llama-standard per-head patching pattern on Llama-8B-Inst, using one trial from `trial_df`. Assertion at the end catches silent slice-write failures.

Reference: `notebooks/unified_pipeline_v4_1*.ipynb`.


In [ ]:
# ── Cell 9: GPU check + load Llama-8B-Inst via nnsight ──
import torch
assert torch.cuda.is_available(), 'GPU required for Part B'
log(f'GPU: {torch.cuda.get_device_name(0)}  VRAM={torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

import nnsight as nns
from nnsight import LanguageModel
log(f'nnsight version: {nns.__version__}')

SMOKE_MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'
t0 = time.time()
model = LanguageModel(SMOKE_MODEL_ID, dtype=torch.bfloat16, device_map='auto')
tokenizer = model.tokenizer
cfg = model.config
arch = {
    'num_layers' : cfg.num_hidden_layers,
    'num_heads'  : cfg.num_attention_heads,
    'head_dim'   : getattr(cfg, 'head_dim', cfg.hidden_size // cfg.num_attention_heads),
    'hidden_size': cfg.hidden_size,
    'vocab_size' : cfg.vocab_size,
}
arch['total_heads'] = arch['num_layers'] * arch['num_heads']
log(f'Loaded {SMOKE_MODEL_ID} in {time.time()-t0:.0f}s')
log(f'arch: {arch}')

smoke = trial_df.iloc[0].to_dict()
log(f'smoke trial: {smoke["relation"]}  subject={smoke["subject"]}  '
    f'target={smoke["target"]}  competitor={smoke["competitor"]}')
log(f'  prompt        : {smoke["prompt"]!r}')
log(f'  corrupt_prompt: {smoke["corrupt_prompt"]!r}')

target_id     = get_token_id_with_space(tokenizer, smoke['target'])
competitor_id = get_token_id_with_space(tokenizer, smoke['competitor'])
log(f'  target_id={target_id}  competitor_id={competitor_id}')


In [ ]:
# ── Cell 10: Clean + collect + patch (v4_1 standard pattern) ──

def clean_logits_last(model, prompt):
    with model.trace(prompt) as tracer:
        logits = model.output.logits[0, -1, :].save()
    return logits.detach().float().cpu().numpy()


def collect_attn_input_per_layer(model, arch, prompt):
    L, H, D = arch['num_layers'], arch['num_heads'], arch['head_dim']
    saves = {}   # init OUTSIDE with-block: nnsight context-mgr can suppress inner exceptions
    with model.trace(prompt) as tracer:
        for i in range(L):
            saves[i] = model.model.layers[i].self_attn.o_proj.input[0, -1, :].save()
    if len(saves) != L:
        raise RuntimeError(f'Trace failed: captured {len(saves)}/{L} layers')
    return np.stack([saves[i].detach().float().cpu().numpy().reshape(H, D) for i in range(L)])


def patched_logits_with_heads(model, arch, prompt, source_vec, patch_heads):
    H, D = arch['num_heads'], arch['head_dim']
    src = torch.tensor(source_vec, dtype=torch.bfloat16, device='cuda')
    sorted_patches = sorted(patch_heads, key=lambda lh: lh[0])
    with model.trace(prompt) as tracer:
        for layer_idx, head_idx in sorted_patches:
            start, end = head_idx * D, (head_idx + 1) * D
            model.model.layers[layer_idx].self_attn.o_proj.input[0, -1, start:end] = src[layer_idx, head_idx]
        logits = model.output.logits[0, -1, :].save()
    return logits.detach().float().cpu().numpy()


clean_lg = clean_logits_last(model, smoke['prompt'])
log(f'Clean argmax={int(clean_lg.argmax())} ({tokenizer.decode([int(clean_lg.argmax())])!r})  '
    f'target_logit={clean_lg[target_id]:.3f}  competitor_logit={clean_lg[competitor_id]:.3f}')

corrupt_vec = collect_attn_input_per_layer(model, arch, smoke['corrupt_prompt'])
assert corrupt_vec.shape == (arch['num_layers'], arch['num_heads'], arch['head_dim'])
log(f'corrupt_vec.shape = {corrupt_vec.shape}')

all_heads = [(l, h) for l in range(arch['num_layers']) for h in range(arch['num_heads'])]
t0 = time.time()
patched_all = patched_logits_with_heads(model, arch, smoke['prompt'], corrupt_vec, all_heads)
log(f'Patched all {len(all_heads)} heads in {time.time()-t0:.1f}s')
log(f'  patched_argmax={int(patched_all.argmax())} ({tokenizer.decode([int(patched_all.argmax())])!r})')
log(f'  target delta  = {patched_all[target_id] - clean_lg[target_id]:+.3f}')

delta_all = float(patched_all[target_id] - clean_lg[target_id])
assert abs(delta_all) > 1e-3, f'Patching had no effect (delta={delta_all}) — slice write may be failing silently'
log(f'SMOKE TEST PASS (delta_all={delta_all:+.3f})')


In [ ]:
# ── Cell 11: Reference patterns for GPT-J and Gemma-3 (printed, not executed) ──
print("""
============================== GPT-J pattern ==============================
# dtype=float32, path=model.transformer.h[L].attn.out_proj, bare trace
def get_logits_with_patch_gptj(model, arch, prompt, source_vec, patch_heads):
    H, D = arch["num_heads"], arch["head_dim"]
    src = torch.tensor(source_vec, dtype=torch.float32, device="cuda")
    for layer_idx, head_idx in sorted(patch_heads, key=lambda lh: lh[0]):
        start, end = head_idx*D, (head_idx+1)*D
        ...  # model.transformer.h[layer_idx].attn.out_proj.input[0, -1, start:end] = src[layer_idx, head_idx]

============================= Gemma-3 pattern =============================
# dtype=bfloat16 (fp16 UNSAFE). trace([prompt]) LIST form required.
# Path: model.model.layers[L].self_attn.o_proj (same as Llama).
with model.trace([prompt]) as tracer:    # <<< list
    ...

=========================== Standard (Llama etc) ==========================
# Llama-8B-Inst, Gemma-2-9B-It, Qwen-2.5-7B/14B-Inst, OLMo-2-13B-Inst
# = the SMOKE TEST above (bare trace, bf16, model.model.layers[L].self_attn.o_proj).
""")
log("Reference patterns printed.")


In [ ]:
# ── Cell 12: Beep + runtime.unassign() ──
try:
    from IPython.display import Audio, display
    sr = 44100
    _t = np.linspace(0, 1, sr)
    display(Audio(0.5 * np.sin(2 * np.pi * 440 * _t), rate=sr, autoplay=True))
except Exception as e:
    log(f'beep failed: {e}')

log('Phase 0 + smoke test complete.')
log(f'Outputs: {OUT_DIR}/')
runtime.unassign()
